# 01 — Boltz-2: Protein–Protein Structure Prediction & Binding Confidence
*Module 05 — `05_structure_prediction/`*

**Purpose:** Run Boltz-2 (Passaro et al. 2025) on phiL7 RBP × Xcc receptor pairs to produce
3D complex structures and ipTM-based binding confidence scores, which seed the cold-start
synthetic prior in Module 06 (deep ensemble, Cycle 0).

**目的：** 使用 Boltz-2 对 phiL7 RBP × Xcc 受体配对进行3D蛋白复合物结构预测，输出
ipTM 绑定置信度分数，作为第0循环模型训练的合成先验数据。

---

**Key references:**
- Passaro S. et al. (2025). *Boltz-2: All-atom protein folding with binding affinity prediction.*
  bioRxiv. https://github.com/jwohlwend/boltz
- Wang W.-T. et al. (2003). TonB-ExbBD complex as phiL7 receptor. *Mol. Microbiol.* 49:1097–1110.

**Boltz-2 affinity head note:** The Boltz-2 affinity head is trained on protein–*small-molecule*
binding data (PDBbind). For protein–protein complexes, `predicted_dG_kcal_mol` and
`predicted_pKd` are set to NaN; we use `ipTM` as the binding confidence proxy instead.
The `interface_ipTM` column in `affinity_priors.csv` carries this value.

**Boltz-2 亲和力注意：** Boltz-2 亲和力头在蛋白-小分子数据上训练，
对蛋白-蛋白复合物不输出 dG/pKd，用 ipTM 替代作为绑定置信度代理。


In [ ]:
# Cell 2 — Imports + version printout
# 导入库并打印版本信息

import json
import logging
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Anchor paths: notebook lives at <module>/processes/
# 路径锚点：notebook 在 <module>/processes/ 下
MODULE_ROOT = Path.cwd().resolve().parent          # 05_structure_prediction/
REPO_ROOT   = MODULE_ROOT.parent                   # repo root

INPUTS_DIR   = MODULE_ROOT / "inputs"
OUTPUTS_DIR  = MODULE_ROOT / "outputs"
BOLTZ_DIR    = OUTPUTS_DIR / "boltz2"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
BOLTZ_DIR.mkdir(parents=True, exist_ok=True)

# Random seeds for reproducibility / 随机种子确保可重复性
np.random.seed(42)

# Version printout / 版本打印
print(f"Python     : {sys.version}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
try:
    import torch; print(f"PyTorch    : {torch.__version__}")
except ImportError:
    print("PyTorch    : not found")

try:
    result = subprocess.run(["boltz", "--help"], capture_output=True, text=True)
    # Extract version from pip show / pip 版本
    vr = subprocess.run(["pip3", "show", "boltz"], capture_output=True, text=True)
    for line in vr.stdout.split("\n"):
        if line.startswith("Version"):
            print(f"Boltz      : {line.split(':')[1].strip()}")
            break
except FileNotFoundError:
    print("Boltz      : not in PATH — run: pip3 install boltz")

import git
try:
    repo = git.Repo(REPO_ROOT, search_parent_directories=True)
    REPO_SHA = repo.head.commit.hexsha[:8]
except Exception:
    REPO_SHA = "unknown"
print(f"Repo SHA   : {REPO_SHA}")


## Method: Boltz-2 Architecture

Boltz-2 (Passaro et al. 2025) is a biomolecular structure prediction model built on
the AlphaFold 2 / AF3 paradigm but fully open-source. It predicts:

1. **Complex structure** — 3D atomic coordinates of the bound complex (PDB / mmCIF output).
2. **Confidence metrics** — pLDDT (per-residue), pTM, and **ipTM** (interface TM-score),
   the latter being our primary binding-confidence proxy for protein–protein pairs.

Unlike the original AlphaFold 3, Boltz-2 adds an explicit affinity head trained on
PDBbind for **protein–small-molecule** systems (outputting dG, pKd, and binary
binder/non-binder classification). For **protein–protein** inputs (used here), the
affinity head does not fire; we rely on ipTM ∈ [0, 1] instead.

**Pipeline role:** The ipTM scores from this notebook become the cold-start prior labels
(`affinity_priors.csv`) consumed by Module 06's deep ensemble during Cycle 0, before
any real ELISA data arrives.

---

**方法说明：**

Boltz-2 是开源的全原子生物分子结构预测模型。对于蛋白-蛋白体系（本节使用），
输出3D复合物结构 + ipTM 置信分数。ipTM ∈ [0, 1] 越高代表预测结合越可靠。
本 notebook 产出的 ipTM 分数作为 Module 06 深度集成的冷启动先验标签。


In [ ]:
# Cell 4 — Helper: prepare Boltz-2 FASTA input for a protein pair
# 辅助函数：为蛋白质对准备 Boltz-2 FASTA 输入文件

def prepare_pair_input(
    rbp_id: str,
    rbp_seq: str,
    receptor_id: str,
    receptor_seq: str,
    work_dir: Path,
) -> Path:
    """Write a Boltz-2 FASTA input file for one (RBP, receptor) pair.
    
    Boltz-2 FASTA format: each chain gets a header >CHAIN_ID|entity_type
    Chain A = RBP, Chain B = receptor (both protein).
    
    Returns the path to the written FASTA file.
    返回写入的 FASTA 文件路径。
    """
    import textwrap
    
    def _wrap60(seq: str) -> str:
        return "\n".join(textwrap.wrap(seq, 60))
    
    pair_name = f"{rbp_id}__{receptor_id}"
    fasta_path = work_dir / f"{pair_name}.fasta"
    
    content = (
        f">A|protein\n{_wrap60(rbp_seq)}\n"
        f">B|protein\n{_wrap60(receptor_seq)}\n"
    )
    fasta_path.write_text(content)
    
    print(f"  Wrote Boltz-2 input: {fasta_path.name}")
    print(f"  Chain A ({rbp_id}): {len(rbp_seq)} aa")
    print(f"  Chain B ({receptor_id}): {len(receptor_seq)} aa")
    return fasta_path


In [ ]:
# Cell 5 — Helper: run Boltz-2 CLI with timeout
# 辅助函数：带超时限制地运行 Boltz-2 CLI

def run_boltz2(
    fasta_path: Path,
    output_dir: Path,
    accelerator: str = "cpu",
    recycling_steps: int = 1,   # default 3; use 1 for CPU speed
    sampling_steps: int = 50,   # default 200; use 50 for CPU speed
    diffusion_samples: int = 1, # minimum
    model: str = "boltz2",
    timeout_s: int = 1800,      # 30-min hard limit for tonight's run
    seed: int = 42,
) -> dict:
    """Invoke boltz predict CLI and return result metadata.
    
    On CPU, a 689-residue complex (85+604 aa) runs in ~15-30 min with
    reduced step counts. If timeout fires, returns partial=True.
    
    在 CPU 上，689 残基复合物（85+604 aa）以降低步数约需 15-30 分钟。
    超时则返回 partial=True，保留输入文件供 Laguna 使用。
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "boltz", "predict", str(fasta_path),
        "--out_dir", str(output_dir),
        "--accelerator", accelerator,
        "--recycling_steps", str(recycling_steps),
        "--sampling_steps", str(sampling_steps),
        "--diffusion_samples", str(diffusion_samples),
        "--model", model,
        "--seed", str(seed),
        "--output_format", "pdb",
        "--write_full_pae",
    ]
    
    print(f"  Command: {' '.join(cmd)}")
    print(f"  Timeout: {timeout_s}s — kill and save config if exceeded.")
    
    start = time.time()
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout_s,
        )
        elapsed = time.time() - start
        success = result.returncode == 0
        if not success:
            print(f"  [WARN] boltz exited with code {result.returncode}")
            print(f"  stderr: {result.stderr[-500:]}")
        else:
            print(f"  [OK] Completed in {elapsed:.0f}s")
    except subprocess.TimeoutExpired:
        elapsed = timeout_s
        success = False
        print(f"  [TIMEOUT] boltz killed after {timeout_s}s — see Laguna runbook.")
        result = type("R", (), {"stdout": "", "stderr": "TIMEOUT", "returncode": -1})()
    
    return {
        "fasta_path":    str(fasta_path),
        "output_dir":    str(output_dir),
        "runtime_s":     elapsed,
        "success":       success,
        "returncode":    result.returncode,
        "partial":       not success,
        "stderr_tail":   result.stderr[-200:] if hasattr(result, "stderr") else "",
    }


In [ ]:
# Cell 6 — Helper: parse Boltz-2 confidence JSON output
# 辅助函数：解析 Boltz-2 输出的置信度 JSON 文件

def parse_boltz2_confidence(output_dir: Path, pair_name: str) -> dict:
    """Parse Boltz-2 confidence JSON from predictions/<pair_name>/ subfolder.
    
    Boltz-2 writes predictions to:
      <out_dir>/boltz_results_<name>/predictions/<name>/confidence_<name>.json
    
    Returns dict with ipTM, pTM, pLDDT, and NaN for protein-protein dG/pKd.
    对于蛋白-蛋白配对，dG 和 pKd 设为 NaN；使用 ipTM 作为置信代理。
    """
    import json as _json
    
    # Boltz-2 output directory layout:
    # <out_dir>/boltz_results_<pair_name>/predictions/<pair_name>/
    results_dir = output_dir / f"boltz_results_{pair_name}"
    pred_dir    = results_dir / "predictions" / pair_name
    
    conf_file = pred_dir / f"confidence_{pair_name}_model_0.json"
    
    if not conf_file.exists():
        # Fallback: search recursively
        candidates = list(output_dir.rglob("confidence_*.json"))
        if candidates:
            conf_file = candidates[0]
        else:
            return {
                "ipTM": float("nan"), "pTM": float("nan"),
                "pLDDT_mean": float("nan"),
                "predicted_dG_kcal_mol": float("nan"),
                "predicted_pKd": float("nan"),
                "confidence": float("nan"),
                "conf_file_found": False,
            }
    
    with conf_file.open() as f:
        data = _json.load(f)
    
    iptm = data.get("iptm", data.get("ipTM", data.get("interface_ptm", float("nan"))))
    ptm  = data.get("ptm", data.get("pTM", float("nan")))
    
    # pLDDT may be per-residue list or a scalar
    plddt_raw = data.get("plddt", data.get("pLDDT", []))
    plddt_mean = float(np.mean(plddt_raw)) if isinstance(plddt_raw, list) and plddt_raw else float(plddt_raw if plddt_raw else float("nan"))
    
    return {
        "ipTM": float(iptm),
        "pTM":  float(ptm),
        "pLDDT_mean": plddt_mean,
        # Protein-protein: no ligand affinity head — set NaN
        # 蛋白-蛋白对无小分子亲和力头，设为 NaN
        "predicted_dG_kcal_mol": float("nan"),
        "predicted_pKd": float("nan"),
        "confidence": float(iptm),   # use ipTM as the confidence column
        "conf_file_found": True,
        "raw_path": str(conf_file),
    }


## Sample Run: phiL7 P25 × Xcc TonB

We run Boltz-2 on the pair:
- **Chain A (RBP):** `EU717894.1_rbp_01` = phiL7 P25 (85 aa) — tail structural protein,
  used as the representative phiL7 RBP candidate for proof-of-concept.
- **Chain B (receptor):** `GCF_000007145.1_tonB` = Xcc XCC3223 / Q8P5W2 (604 aa) —
  the outer-membrane TonB complex component identified by Wang et al. 2003 as the
  phiL7 receptor.

CPU runtime estimate (reduced steps): **15–30 min**. Hard timeout: **30 min**.
If timeout fires, the input FASTA is saved for the Laguna HPC batch run.

---

**运行样例：** phiL7 P25 (85 aa) × Xcc TonB (604 aa, Q8P5W2)。
在 CPU 上，降低步数设定（recycling=1, sampling=50）估计运行 15-30 分钟。
超时自动保存输入文件，供 Laguna HPC 调用。


In [ ]:
# Cell 8 — Load sequences and run sample pair
# 加载序列并运行样本配对

from Bio import SeqIO

RBP_FASTA    = INPUTS_DIR / "EU717894.1_rbp_sequences.faa"
RECEPT_FASTA = INPUTS_DIR / "xcc_receptors_minimal.faa"

# Load RBP sequences / 加载 RBP 序列
rbp_records = {r.id.split("|")[0].strip(): str(r.seq)
               for r in SeqIO.parse(RBP_FASTA, "fasta")}

# Load receptor sequences / 加载受体序列
rec_records = {r.id.split("|")[0].strip(): str(r.seq)
               for r in SeqIO.parse(RECEPT_FASTA, "fasta")}

print("RBP candidates:", list(rbp_records.keys()))
print("Receptors:", list(rec_records.keys()))

# Define the pair for today's smoke test / 定义今晚的测试配对
RBP_ID      = "EU717894.1_rbp_01"
RECEPTOR_ID = "GCF_000007145.1_tonB"

rbp_seq = rbp_records.get(RBP_ID)
rec_seq = rec_records.get(RECEPTOR_ID)

assert rbp_seq, f"RBP sequence not found: {RBP_ID}"
assert rec_seq, f"Receptor sequence not found: {RECEPTOR_ID}"
print(f"\nPair: {RBP_ID} ({len(rbp_seq)} aa) × {RECEPTOR_ID} ({len(rec_seq)} aa)")
print(f"Total residues: {len(rbp_seq) + len(rec_seq)}")


In [ ]:
# Cell 9 — Prepare input and launch Boltz-2 (30-min timeout)
# 准备输入并启动 Boltz-2（30分钟超时）

PAIR_NAME  = f"{RBP_ID}__{RECEPTOR_ID}"
PAIR_OUTDIR = BOLTZ_DIR / PAIR_NAME
WORK_DIR    = INPUTS_DIR  # store the fasta input here too

# Write FASTA input / 写 FASTA 输入
fasta_path = prepare_pair_input(RBP_ID, rbp_seq, RECEPTOR_ID, rec_seq, INPUTS_DIR)

# Launch Boltz-2 / 启动 Boltz-2
print("\nStarting Boltz-2 prediction — timeout=1800s (30 min) ...")
run_meta = run_boltz2(
    fasta_path=fasta_path,
    output_dir=PAIR_OUTDIR,
    accelerator="cpu",
    recycling_steps=1,   # speed optimisation for CPU / CPU 加速设置
    sampling_steps=50,   # reduced from default 200 / 从默认200降至50
    diffusion_samples=1,
    timeout_s=1800,
    seed=42,
)
print("\nRun metadata:")
for k, v in run_meta.items():
    print(f"  {k}: {v}")


In [ ]:
# Cell 10 — Parse confidence output and append to affinity_priors.csv
# 解析置信度输出并追加至 affinity_priors.csv

AFFINITY_CSV = OUTPUTS_DIR / "affinity_priors.csv"

conf = parse_boltz2_confidence(PAIR_OUTDIR, PAIR_NAME)
print("Confidence metrics:", conf)

# Build one row matching INTERFACE §Module 05 schema
# 构建符合 INTERFACE §Module 05 模式的单行数据
row = {
    "rbp_id":               RBP_ID,
    "receptor_id":          RECEPTOR_ID,
    "model":                "boltz2_2.0.3",
    "predicted_dG_kcal_mol": conf["predicted_dG_kcal_mol"],   # NaN for protein-protein
    "predicted_pKd":         conf["predicted_pKd"],            # NaN for protein-protein
    "confidence":            conf["confidence"],                # ipTM
    "interface_ipTM":        conf["ipTM"],
    "pTM":                   conf.get("pTM", float("nan")),
    "pLDDT_mean":            conf.get("pLDDT_mean", float("nan")),
    "run_success":           run_meta["success"],
    "runtime_s":             run_meta["runtime_s"],
    "partial":               run_meta["partial"],
}

df_new = pd.DataFrame([row])

if AFFINITY_CSV.exists():
    df_existing = pd.read_csv(AFFINITY_CSV)
    # Remove duplicate if re-running / 去重
    df_existing = df_existing[df_existing["rbp_id"] != RBP_ID or df_existing["receptor_id"] != RECEPTOR_ID]
    df = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df = df_new

df.to_csv(AFFINITY_CSV, index=False, float_format="%.6f")
print(f"\nSaved {len(df)} row(s) to {AFFINITY_CSV}")
print(df[["rbp_id", "receptor_id", "confidence", "interface_ipTM", "partial"]].to_string())


In [ ]:
# Cell 11 — Sanity assertions (fail fast if data is malformed)
# 合理性断言（数据格式错误时快速失败）

import ast

# 1. Output CSV exists and has required columns
# CSV 存在且含所有必需列
required_cols = [
    "rbp_id", "receptor_id", "model",
    "predicted_dG_kcal_mol", "predicted_pKd",
    "confidence", "interface_ipTM",
]
df_check = pd.read_csv(AFFINITY_CSV)
for col in required_cols:
    assert col in df_check.columns, f"Missing column: {col}"
print("[OK] affinity_priors.csv columns verified")

# 2. If run completed, PDB should exist
# 如果运行成功，PDB 文件应存在
if not run_meta["partial"]:
    pdb_candidates = list(PAIR_OUTDIR.rglob("*.pdb"))
    assert pdb_candidates, f"No PDB found in {PAIR_OUTDIR}"
    # Verify PDB has ATOM records / 验证 PDB 含 ATOM 记录
    pdb_text = pdb_candidates[0].read_text()
    assert "ATOM" in pdb_text, f"PDB has no ATOM records: {pdb_candidates[0]}"
    print(f"[OK] PDB found and contains ATOM records: {pdb_candidates[0].name}")
else:
    print("[INFO] Run partial/timeout — PDB check skipped; FASTA input saved for Laguna")

# 3. Sequences are intact in input files
# 输入文件序列完整性检查
assert RBP_FASTA.exists(), f"RBP FASTA missing: {RBP_FASTA}"
assert RECEPT_FASTA.exists(), f"Receptor FASTA missing: {RECEPT_FASTA}"
print("[OK] Input FASTA files present")

print("\nAll sanity checks passed.")


In [ ]:
# Cell 12 — (Optional) second pair: P25 × ExbB to show batching pattern
# （可选）第二配对：P25 × ExbB，演示批处理模式

PAIR2_RBP_ID   = "EU717894.1_rbp_01"
PAIR2_REC_ID   = "GCF_000007145.1_exbB"
PAIR2_NAME     = f"{PAIR2_RBP_ID}__{PAIR2_REC_ID}"

rec2_seq = rec_records.get(PAIR2_REC_ID)
print(f"Second pair: {PAIR2_RBP_ID} ({len(rbp_seq)} aa) × {PAIR2_REC_ID} ({len(rec2_seq)} aa)")
print("NOTE: This cell is for documentation only tonight.")
print("      For the actual Laguna batch, see AGENT_REPORT.md §Laguna Runbook.")
print("      Skip running to stay within 30-min CPU budget.")

# The input FASTA for pair 2 is written here but NOT predicted tonight.
# 第二配对的 FASTA 文件写好但今晚不预测，Laguna 集群上运行。
fasta2 = prepare_pair_input(PAIR2_RBP_ID, rbp_seq, PAIR2_REC_ID, rec2_seq, INPUTS_DIR)
print(f"\nInput written for Laguna: {fasta2}")
